# 19. 3D point-cloud and detection — paper-style small pipelines

Only point count, channel width, and BEV grid size are reduced. The two earlier shortcuts are removed:

- PointNet++ uses **FPS + metric-radius ball query + local PointNet + max pooling**, not kNN substitution.
- The BEV path uses a **PointPillars-style pillar feature network** with point, cluster-offset, and pillar-center features before max aggregation, instead of directly averaging point features.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
device = torch.device("cpu")
print("device:", device)


## 1. PointNet++ sampling and ball query


In [ ]:
def farthest_point_sampling(xyz, count):
    point_count = xyz.size(0)
    selected = torch.zeros(count, dtype=torch.long, device=xyz.device)
    minimum_distance = torch.full(
        (point_count,),
        float("inf"),
        device=xyz.device,
    )
    farthest = torch.tensor(0, device=xyz.device)

    for sample_index in range(count):
        selected[sample_index] = farthest
        centroid = xyz[farthest : farthest + 1]
        distance = (xyz - centroid).square().sum(dim=-1)
        minimum_distance = torch.minimum(minimum_distance, distance)
        farthest = minimum_distance.argmax()

    return selected


def ball_query(query_xyz, source_xyz, radius, samples):
    distances = torch.cdist(query_xyz, source_xyz)
    grouped_indices = []

    for distance_row in distances:
        valid = torch.where(distance_row <= radius)[0]
        if valid.numel() == 0:
            valid = distance_row.argmin().view(1)

        chosen = valid[:samples]
        if chosen.numel() < samples:
            padding = chosen[-1:].repeat(samples - chosen.numel())
            chosen = torch.cat([chosen, padding])

        grouped_indices.append(chosen)

    return torch.stack(grouped_indices)


## 2. Hierarchical PointNet++ set abstraction


In [ ]:
class SetAbstraction(nn.Module):
    def __init__(self, input_feature_dim, mlp_dims):
        super().__init__()

        layers = []
        input_dim = 3 + input_feature_dim
        for output_dim in mlp_dims:
            layers.append(nn.Linear(input_dim, output_dim))
            layers.append(nn.ReLU())
            input_dim = output_dim

        self.local_pointnet = nn.Sequential(*layers)

    def forward(
        self,
        source_xyz,
        source_features,
        centroid_count,
        radius,
        neighbors,
    ):
        centroid_ids = farthest_point_sampling(
            source_xyz,
            centroid_count,
        )
        centroid_xyz = source_xyz[centroid_ids]
        neighbor_ids = ball_query(
            centroid_xyz,
            source_xyz,
            radius,
            neighbors,
        )

        relative_xyz = (
            source_xyz[neighbor_ids]
            - centroid_xyz[:, None, :]
        )

        if source_features is None:
            local_input = relative_xyz
        else:
            neighbor_features = source_features[neighbor_ids]
            local_input = torch.cat(
                [relative_xyz, neighbor_features],
                dim=-1,
            )

        local_features = self.local_pointnet(local_input)
        pooled = local_features.max(dim=1).values
        return centroid_xyz, pooled


class TinyPointNetPlusPlus(nn.Module):
    def __init__(self, classes=4):
        super().__init__()
        self.sa1 = SetAbstraction(0, [8, 8, 16])
        self.sa2 = SetAbstraction(16, [16, 16, 24])
        self.classifier = nn.Linear(24, classes)

    def forward(self, xyz):
        xyz1, features1 = self.sa1(
            xyz,
            None,
            centroid_count=12,
            radius=0.50,
            neighbors=8,
        )
        _, features2 = self.sa2(
            xyz1,
            features1,
            centroid_count=4,
            radius=0.90,
            neighbors=8,
        )
        global_feature = features2.max(dim=0).values
        return self.classifier(global_feature[None])


## 3. PointNet++ five-step CPU check


In [ ]:
points = torch.rand(32, 3, device=device)
label = torch.tensor([2], device=device)

pointnet_pp = TinyPointNetPlusPlus().to(device)
optimizer = torch.optim.Adam(pointnet_pp.parameters(), lr=3e-3)

pointnet_loss_history = []
for step in range(5):
    optimizer.zero_grad()
    logits = pointnet_pp(points)
    loss = F.cross_entropy(logits, label)
    loss.backward()
    optimizer.step()

    pointnet_loss_history.append(loss.item())
    print(f"PointNet++ step {step + 1}: loss={loss.item():.6f}")

print("PointNet++ loss history:", pointnet_loss_history)


## 4. DGCNN EdgeConv and Point Transformer local attention


In [ ]:
point_features = nn.Sequential(
    nn.Linear(3, 16),
    nn.ReLU(),
    nn.Linear(16, 16),
).to(device)(points)

pairwise = torch.cdist(points, points)
knn_ids = pairwise.topk(k=3, largest=False).indices[:, 1:]
center = points[:, None].expand(-1, knn_ids.size(1), -1)
neighbor = points[knn_ids]

edge_input = torch.cat([center, neighbor - center], dim=-1)
edge_mlp = nn.Sequential(
    nn.Linear(6, 16),
    nn.ReLU(),
    nn.Linear(16, 16),
).to(device)
edge_features = edge_mlp(edge_input).max(dim=1).values

q_proj = nn.Linear(16, 16).to(device)
k_proj = nn.Linear(16, 16).to(device)
v_proj = nn.Linear(16, 16).to(device)
pos_mlp = nn.Sequential(
    nn.Linear(3, 16),
    nn.ReLU(),
    nn.Linear(16, 16),
).to(device)
attn_mlp = nn.Sequential(
    nn.Linear(16, 16),
    nn.ReLU(),
    nn.Linear(16, 16),
).to(device)

q = q_proj(point_features)
k = k_proj(point_features)[knn_ids]
v = v_proj(point_features)[knn_ids]
relative = points[:, None] - points[knn_ids]
positional = pos_mlp(relative)
attention_logits = attn_mlp(q[:, None] - k + positional)
attention_weights = attention_logits.softmax(dim=1)
point_transformer = (
    attention_weights * (v + positional)
).sum(dim=1)

print("EdgeConv:", edge_features.shape)
print("Point Transformer:", point_transformer.shape)


## 5. PointPillars-style pillar feature network

Each point carries raw coordinates/intensity, offset from the pillar's point mean, and offset from the geometric pillar center. A pointwise linear layer is followed by max aggregation into the BEV pseudo-image.


In [ ]:
class PillarFeatureNet(nn.Module):
    def __init__(self, output_channels=16, voxel_size=0.5):
        super().__init__()
        self.output_channels = output_channels
        self.voxel_size = voxel_size
        self.point_linear = nn.Linear(9, output_channels, bias=False)

    def forward(self, xyz, intensity):
        raw_voxel_ids = torch.floor(
            xyz[:, :2] / self.voxel_size
        ).long()
        minimum_id = raw_voxel_ids.min(dim=0).values
        local_voxel_ids = raw_voxel_ids - minimum_id

        height = int(local_voxel_ids[:, 1].max().item()) + 1
        width = int(local_voxel_ids[:, 0].max().item()) + 1
        bev = torch.zeros(
            self.output_channels,
            height,
            width,
            device=xyz.device,
        )

        unique_ids = torch.unique(local_voxel_ids, dim=0)
        for local_id in unique_ids:
            mask = (local_voxel_ids == local_id).all(dim=1)
            pillar_xyz = xyz[mask]
            pillar_intensity = intensity[mask, None]

            cluster_offset = (
                pillar_xyz - pillar_xyz.mean(dim=0, keepdim=True)
            )

            raw_id = local_id + minimum_id
            pillar_center_xy = (
                raw_id.float() + 0.5
            ) * self.voxel_size
            center_offset_xy = (
                pillar_xyz[:, :2] - pillar_center_xy
            )

            point_input = torch.cat(
                [
                    pillar_xyz,
                    pillar_intensity,
                    cluster_offset,
                    center_offset_xy,
                ],
                dim=-1,
            )
            point_hidden = F.relu(self.point_linear(point_input))
            pillar_feature = point_hidden.max(dim=0).values

            x_id = int(local_id[0].item())
            y_id = int(local_id[1].item())
            bev[:, y_id, x_id] = pillar_feature

        return bev


## 6. CenterPoint-style BEV backbone and heads


In [ ]:
class TinyCenterPoint(nn.Module):
    def __init__(self, input_channels=16, hidden=24):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(input_channels, hidden, 3, padding=1),
            nn.BatchNorm2d(hidden),
            nn.ReLU(),
            nn.Conv2d(hidden, hidden, 3, padding=1),
            nn.ReLU(),
        )
        self.heatmap = nn.Conv2d(hidden, 1, 1)
        self.offset = nn.Conv2d(hidden, 2, 1)
        self.height = nn.Conv2d(hidden, 1, 1)
        self.dimensions = nn.Conv2d(hidden, 3, 1)
        self.rotation = nn.Conv2d(hidden, 2, 1)

    def forward(self, x):
        hidden = self.backbone(x)
        return {
            "heatmap_logits": self.heatmap(hidden),
            "offset": self.offset(hidden),
            "height": self.height(hidden),
            "dimensions": F.softplus(self.dimensions(hidden)),
            "rotation": self.rotation(hidden),
        }


def decode_centerpoint(prediction, voxel_size=0.5):
    heatmap = torch.sigmoid(prediction["heatmap_logits"])
    heatmap = heatmap * (
        F.max_pool2d(heatmap, 3, stride=1, padding=1) == heatmap
    )

    batch, _, height, width = heatmap.shape
    score, flat_index = heatmap.view(batch, -1).max(dim=-1)
    y = torch.div(flat_index, width, rounding_mode="floor")
    x = flat_index % width
    batch_ids = torch.arange(batch, device=heatmap.device)

    offset = prediction["offset"][batch_ids, :, y, x]
    z = prediction["height"][batch_ids, 0, y, x]
    dimensions = prediction["dimensions"][batch_ids, :, y, x]
    rotation = prediction["rotation"][batch_ids, :, y, x]
    yaw = torch.atan2(rotation[:, 0], rotation[:, 1])

    center_x = (x.float() + offset[:, 0]) * voxel_size
    center_y = (y.float() + offset[:, 1]) * voxel_size
    box = torch.cat(
        [
            center_x[:, None],
            center_y[:, None],
            z[:, None],
            dimensions,
            yaw[:, None],
        ],
        dim=-1,
    )
    return score, box


## 7. Pillar + CenterPoint five-step CPU check


In [ ]:
lidar_points = torch.rand(24, 3, device=device) * 2.0
intensity = torch.rand(24, device=device)

pillar_encoder = PillarFeatureNet().to(device)
centerpoint = TinyCenterPoint().to(device)
optimizer = torch.optim.AdamW(
    list(pillar_encoder.parameters()) + list(centerpoint.parameters()),
    lr=2e-3,
)

centerpoint_loss_history = []
for step in range(5):
    optimizer.zero_grad()

    bev = pillar_encoder(lidar_points, intensity)[None]
    prediction = centerpoint(bev)

    target_heatmap = torch.zeros_like(prediction["heatmap_logits"])
    target_heatmap[:, :, 1, 1] = 1.0

    loss = F.binary_cross_entropy_with_logits(
        prediction["heatmap_logits"],
        target_heatmap,
    )
    loss = loss + 0.05 * prediction["offset"].square().mean()
    loss = loss + 0.05 * (prediction["dimensions"] - 1.0).square().mean()
    loss = loss + 0.05 * prediction["height"].square().mean()
    loss = loss + 0.05 * prediction["rotation"].square().mean()

    loss.backward()
    optimizer.step()

    centerpoint_loss_history.append(loss.item())
    print(f"CenterPoint step {step + 1}: loss={loss.item():.6f}")

score, box = decode_centerpoint(centerpoint(pillar_encoder(lidar_points, intensity)[None]))
print("CenterPoint loss history:", centerpoint_loss_history)
print("decoded [x,y,z,w,l,h,yaw]:", box)


## References and provenance

- PointNet++: hierarchical farthest-point sampling, metric-radius ball query, local PointNet, and symmetric max aggregation.
- DGCNN: EdgeConv over local neighbors.
- Point Transformer: vector attention with relative 3D position.
- PointPillars: augmented point features followed by pointwise feature extraction and per-pillar max pooling into a BEV pseudo-image.
- CenterPoint: center heatmap and regression heads for 3D center, dimensions, height, and rotation.

Only tensor sizes and point/grid counts are reduced.
